In [ ]:
import pandas as pd
from pathlib import Path

# ============================================================
# SETUP
# ============================================================

# Change this to the name/path of your original Excel file.
input_file = Path("Reading Log 2016-today1.xlsx")

# Name of the new Excel file that will be created.
output_file = Path("wordpress_book_database_import.xlsx")


# ============================================================
# 1. READ ALL YEAR TABS
# ============================================================

excel_file = pd.ExcelFile(input_file)
all_years = []

for sheet_name in excel_file.sheet_names:

    # Only process sheets whose names are years, e.g. 2016, 2017, 2025, 2026.
    try:
        year = int(str(sheet_name).strip())
    except ValueError:
        print(f"Skipping non-year sheet: {sheet_name}")
        continue

    df = pd.read_excel(
        input_file,
        sheet_name=sheet_name
    )

    # Remove completely empty rows.
    df = df.dropna(how="all").copy()

    if df.empty:
        continue

    # Remove all spaces from the original column headers.
    df.columns = (
        df.columns
        .astype(str)
        .str.replace(r'\s+', '', regex=True)
    )

    # Treat "Title" and "BookTitle" as the same field.
    if "Title" in df.columns and "BookTitle" not in df.columns:
        df = df.rename(columns={"Title": "BookTitle"})


    # Remember which year the row came from.
    df["_sheet_year"] = year

    all_years.append(df)


if not all_years:
    raise ValueError(
        "No year-named worksheets containing data were found."
    )


reading_log = pd.concat(
    all_years,
    ignore_index=True,
    sort=False
)


# ============================================================
# 2. CLEANING
# ============================================================

# Remove all spaces from column headers again after combining.
# This also makes sure every year tab has consistent headers.

reading_log.columns = (
    reading_log.columns
    .astype(str)
    .str.replace(r'\s+', '', regex=True)
)

def clean_text(value):
    if pd.isna(value):
        return ""

    return str(value).strip()

def clean_author_name(value):
    if pd.isna(value):
        return ""

    value = str(value).strip()

    # Remove trailing asterisks and spaces.
    value = value.rstrip("*").rstrip()

    # Standardize editor notation.
    if value.endswith("(Editor)"):
        value = value[:-8].rstrip() + "(Ed.)"
    elif value.endswith("(Ed)"):
        value = value[:-4].rstrip() + "(Ed.)"

    # If there is a comma, reverse the name.
    if "," in value:
        last_name, first_name = value.split(",", 1)
        value = first_name.strip() + " " + last_name.strip()

    return value.rstrip()

def clean_date(value):
    # Finished dates are stored as YYYY-MM-DD.
    # Keep them in exactly that format.

    if pd.isna(value) or str(value).strip() == "":
        return ""

    return str(value).strip()


def clean_bool(value):
    # Convert common yes/no values to 1/0.
    # Blank remains blank.

    if pd.isna(value) or str(value).strip() == "":
        return ""

    value = str(value).strip().lower()

    if value in ("yes", "y", "true", "1"):
        return "1"

    if value in ("no", "n", "false", "0"):
        return "0"

    return str(value).strip()

def combine_countries(birth, current):
    countries = []

    for value in [birth, current]:
        if pd.isna(value) or str(value).strip() == "":
            continue

        # Split existing comma-separated values
        for country in str(value).split(","):
            country = country.strip()

            if country and country not in countries:
                countries.append(country)

    return ", ".join(countries)

# ============================================================
# 4. CLEAN THE MAIN DATA
# ============================================================

# Finished and Started are dates and are always stored as YYYY-MM-DD.
reading_log["Finished"] = (
    reading_log["Finished"]
    .astype(str)
    .str.strip()
)

text_columns = [
    "book_id",
    "BookTitle",
    "ISBN"
    "Pages",
    "Format",
    "Fiction/Nonfic",
    "Front/Back",
    "Genre",
    "OriginalLanguage",
    "LanguageRead",
    "Author",
    "BirthCountry",
    "CurrentCountry",
    "CountryDetails",
    "Gender",
    "BIPOC",
    "LGBT",
    "Disabled/Neurodiverse",
    "Read",
    "Rating",
]

for column in text_columns:

    if column in reading_log.columns:
        reading_log[column] = (
            reading_log[column]
            .apply(clean_text)
        )

reading_log["Author"] = reading_log["Author"].apply(clean_author_name)

# ============================================================
# 5. AUTHORS
# ============================================================

# Each unique author becomes one Author object.

authors = reading_log[
    reading_log["Author"].str.strip() != ""
].copy()


authors = authors.drop_duplicates(
    subset=["Author"],
    keep="first"
).copy()


# Create the fields corresponding to your WordPress Author object.

authors_output = pd.DataFrame({

    "Author": authors["Author"],
    "Country": [
        combine_countries(birth, current)
        for birth, current in zip(
            authors["BirthCountry"],
            authors["CurrentCountry"]
        )
    ],
    "CountryDetails": authors["CountryDetails"],
    "Gender": authors["Gender"],
    "BIPOC": authors["BIPOC"].apply(clean_bool),
    "LGBTQ+": authors["LGBT"].apply(clean_bool),
    "Disabled/Neurodiverse": authors["Disabled/Neurodiverse"].apply(clean_bool),
})


# ============================================================
# 6. CREATE AN INTERNAL AUTHOR ID
# ============================================================

# WordPress will create its own Author post IDs when imported.
#
# These temporary IDs simply allow the Books sheet to refer
# consistently to the correct author.

author_lookup = {}

for index, row in authors_output.iterrows():
    author_name = row["Author"]
    author_lookup[author_name] = index + 1

authors_output.insert(
    0,
    "ImportAuthorID",
    authors_output.index.map(
        lambda x: x + 1
    )
)

# ============================================================
# 7. BOOKS
# ============================================================

# A book is identified primarily by book_id when available.
# If book_id is missing, Book Title + Author is used.

reading_log["_book_key"] = ""

has_book_id = (
    reading_log["book_id"].str.strip() != ""
)

reading_log.loc[
    has_book_id,
    "_book_key"
] = (
    "ID:"
    + reading_log.loc[
        has_book_id,
        "book_id"
    ].str.strip()
)

missing_id = ~has_book_id

reading_log.loc[
    missing_id,
    "_book_key"
] = (
    "TITLE:"
    + reading_log.loc[
        missing_id,
        "BookTitle"
    ].str.strip().str.lower()
    + "|AUTHOR:"
    + reading_log.loc[
        missing_id,
        "Author"
    ].str.strip().str.lower()
)


# Keep the first occurrence of each unique book.

books = reading_log[
    reading_log["_book_key"] != ""
].drop_duplicates(
    subset=["_book_key"],
    keep="first"
).copy()


# Create temporary Import Book IDs.

books_output = pd.DataFrame({

    "ImportBookID": range(
        1,
        len(books) + 1
    ),

    "BookTitle": books["BookTitle"].values,

    "Author": books["Author"].values,

    "ImportAuthorID": [
        author_lookup.get(
            author,
            ""
        )
        for author in books["Author"]
    ],

    "Pages": books["Pages"].values,
    "Format": books["Format"].values,
    "Fiction/Nonfiction": books["Fiction/Nonfic"].values,
    "Front/Back": books["Front/Back"].values,
    "Genre": books["Genre"].values,
    "OriginalLanguage": books["OriginalLanguage"].values,

    # Keep the ISBN/book_id available for the Open Library step.
    "ISBN": books["ISBN"].values,
})


# ============================================================
# 8. CREATE LOOKUP FROM BOOK KEY TO IMPORT BOOK ID
# ============================================================

book_key_to_import_id = dict(
    zip(
        books["_book_key"],
        books_output["ImportBookID"]
    )
)

# ============================================================
# 9. READ INSTANCES
# ============================================================

# Every row in the original log represents a reading event.
# Therefore, every row becomes a Read Instance.

read_instances = reading_log.copy()

read_instances_output = pd.DataFrame({

    "Book": read_instances["_book_key"].map(
        book_key_to_import_id),
    "BookTitle": read_instances["BookTitle"],
    "DateRead": read_instances["Finished"].astype(str).str.strip(),
    "Rating": read_instances["Rating"],
    "Format": read_instances["Format"],
    "LanguageRead": read_instances["LanguageRead"],
    "Year": read_instances["_sheet_year"],
    "Read": read_instances["Read"],
})

# ============================================================
# 10. REMOVE COMPLETELY EMPTY READ INSTANCES
# ============================================================

read_instances_output = read_instances_output[
    read_instances_output["BookTitle"].str.strip() != ""
].copy()

# ============================================================
# 11. REMOVE ALL SPACES FROM OUTPUT COLUMN HEADERS
# ============================================================

authors_output.columns = (
    authors_output.columns
    .astype(str)
    .str.replace(r'\s+', '', regex=True)
)

books_output.columns = (
    books_output.columns
    .astype(str)
    .str.replace(r'\s+', '', regex=True)
)

read_instances_output.columns = (
    read_instances_output.columns
    .astype(str)
    .str.replace(r'\s+', '', regex=True)
)


# ============================================================
# 12. EXPORT EVERYTHING TO ONE EXCEL FILE
# ============================================================

with pd.ExcelWriter(
    output_file,
    engine="openpyxl"
) as writer:

    authors_output.to_excel(
        writer,
        sheet_name="Authors",
        index=False
    )

    books_output.to_excel(
        writer,
        sheet_name="Books",
        index=False
    )

    read_instances_output.to_excel(
        writer,
        sheet_name="Read Instances",
        index=False
    )


# ============================================================
# 13. SUMMARY
# ============================================================

print(f"Authors:{len(authors_output)}")
print(f"Books:{len(books_output)}")
print(f"Read Instances: {len(read_instances_output)}")
print(f"Output file: {output_file.resolve()}")

In [ ]:
import requests
import time
from pathlib import Path

# Read the Books sheet from the file we just created.
books_for_lookup = pd.read_excel(
    output_file,
    sheet_name="Books",
    dtype=str
).fillna("")

print(f"Books in spreadsheet: {len(books_for_lookup)}")

# Add fields if they don't already exist

if "ISBN" not in books_for_lookup.columns:
    books_for_lookup["ISBN"] = ""

if "PublicationYear" not in books_for_lookup.columns:
    books_for_lookup["PublicationYear"] = ""

if "Pages" not in books_for_lookup.columns:
    books_for_lookup["Pages"] = ""

if "OpenLibraryStatus" not in books_for_lookup.columns:
    books_for_lookup["OpenLibraryStatus"] = ""


# ------------------------------------------------------------
# Open Library session
# ------------------------------------------------------------

session = requests.Session()

session.headers.update({
    "User-Agent": "BookDatabase/1.0 (personal book database)"
})

# Search Open Library

def search_open_library(title, author):

    title = str(title).strip()
    author = str(author).strip()

    if not title:
        return []

    # Ignore subtitles when searching.
    if ":" in title:
        search_title = title.split(":", 1)[0].strip()
    else:
        search_title = title

    # Remove editor notation.
    search_author = (
        author
        .replace("(Ed.)", "")
        .replace("(Editor)", "")
        .strip()
    )

    fields = (
        "key,title,author_name,first_publish_year,"
        "isbn,number_of_pages_median"
    )

    # First search: title + author

    params = {
        "title": search_title,
        "author": search_author,
        "limit": 5,
        "fields": fields
    }

    try:
        response = session.get(
            "https://openlibrary.org/search.json",
            params=params,
            timeout=8
        )

        response.raise_for_status()

        results = response.json().get("docs", [])

        if results:
            return results

    except requests.RequestException as e:
        print(f"  Search error: {e}")


    # Second search: title only

    params = {
        "title": search_title,
        "limit": 5,
        "fields": fields
    }

    try:
        response = session.get(
            "https://openlibrary.org/search.json",
            params=params,
            timeout=8
        )

        response.raise_for_status()

        results = response.json().get("docs", [])

    except requests.RequestException as e:
        print(f"  Title search error: {e}")
        return []


    # Try to find a title result with a matching author.
    search_author_lower = search_author.lower().strip()

    for result in results:

        result_authors = result.get("author_name", [])

        for result_author in result_authors:

            result_author_lower = (
                str(result_author)
                .lower()
                .strip()
            )

            if (
                search_author_lower == result_author_lower
                or search_author_lower in result_author_lower
                or result_author_lower in search_author_lower
            ):
                return [result]

    # If we couldn't match the author, return the title results.
    return results


# ------------------------------------------------------------
# Save progress function
# ------------------------------------------------------------

def save_books_progress():

    progress_file = Path("wordpress_book_database_progress.xlsx")

    with pd.ExcelWriter(
        progress_file,
        engine="openpyxl"
    ) as writer:

        authors_output.to_excel(
            writer,
            sheet_name="Authors",
            index=False
        )

        books_for_lookup.to_excel(
            writer,
            sheet_name="Books",
            index=False
        )

        read_instances_output.to_excel(
            writer,
            sheet_name="Read Instances",
            index=False
        )

    return progress_file


# ------------------------------------------------------------
# Open Library lookups
# ------------------------------------------------------------

total = len(books_for_lookup)

for count, (index, row) in enumerate(
    books_for_lookup.iterrows(),
    start=1
):

    # Skip books we've already processed.
    status = str(
        row.get("OpenLibraryStatus", "")
    ).strip()

    if status in ["Found", "Not Found"]:
        continue


    title = str(row["BookTitle"]).strip()
    author = str(row["Author"]).strip()

    # Search Open Library

    results = search_open_library(title, author)


    # No result

    if not results:

        books_for_lookup.at[
            index,
            "OpenLibraryStatus"
        ] = "Not Found"

        continue


    # Use first result

    result = results[0]


    # ISBN

    if not str(row["ISBN"]).strip():

        isbns = result.get("isbn", [])

        if isbns:
            books_for_lookup.at[
                index,
                "ISBN"
            ] = str(isbns[0])


    # Publication Year

    if not str(row["PublicationYear"]).strip():

        publication_year = result.get(
            "first_publish_year"
        )

        if publication_year:

            books_for_lookup.at[
                index,
                "PublicationYear"
            ] = str(publication_year)


    # Pages

    if not str(row["Pages"]).strip():

        pages = result.get(
            "number_of_pages_median"
        )

        if pages:

            books_for_lookup.at[
                index,
                "Pages"
            ] = str(pages)


    # Status

    books_for_lookup.at[
        index,
        "OpenLibraryStatus"
    ] = "Found"


    # Small pause between requests
    time.sleep(0.2)


    # Save every 25 books

    if count % 25 == 0:

        save_books_progress()

        print(
            f"  Progress saved after {count} books."
        )


# ------------------------------------------------------------
# Final save
# ------------------------------------------------------------

output_file = Path(
    "wordpress_book_database_import.xlsx"
)

with pd.ExcelWriter(
    output_file,
    engine="openpyxl"
) as writer:

    authors_output.to_excel(
        writer,
        sheet_name="Authors",
        index=False
    )

    books_for_lookup.to_excel(
        writer,
        sheet_name="Books",
        index=False
    )

    read_instances_output.to_excel(
        writer,
        sheet_name="Read Instances",
        index=False
    )

# Takes 13 min to print
print("Open Library lookup complete.")
print(f"Output saved to: {output_file}")

In [ ]:
# Update that Open Library file with Goodreads export

import pandas as pd
import re
from pathlib import Path
from openpyxl import load_workbook

# FILES

input_file = Path("wordpress_book_database_import.xlsx")
goodreads_file = Path("goodreads_library_export.csv")
output_file = Path("wordpress_book_database_import_goodreads.xlsx")


# ============================================================
# HELPER: NORMALIZE TITLES FOR MATCHING
# ============================================================

def normalize_title(title):
    """
    Normalize a book title so minor differences between
    Goodreads and the import workbook don't prevent matching.
    """

    if pd.isna(title):
        return ""

    title = str(title).strip().lower()

    # Replace curly apostrophes/quotes with regular characters
    title = title.replace("’", "'")
    title = title.replace("‘", "'")
    title = title.replace("“", '"')
    title = title.replace("”", '"')

    # Remove text in brackets/parentheses
    # Uncomment this if you want editions/subtitles in parentheses ignored.
    # title = re.sub(r"\([^)]*\)", "", title)

    # Replace punctuation with spaces
    title = re.sub(r"[^\w\s]", " ", title)

    # Collapse multiple spaces
    title = re.sub(r"\s+", " ", title).strip()

    return title


# ============================================================
# READ THE WORKBOOKS
# ============================================================


books = pd.read_excel(
    input_file,
    sheet_name="Books",
    dtype=str
).fillna("")

read_instances = pd.read_excel(
    input_file,
    sheet_name="Read Instances",
    dtype=str
).fillna("")

authors = pd.read_excel(
    input_file,
    sheet_name="Authors",
    dtype=str
).fillna("")


goodreads = pd.read_csv(
    goodreads_file,
    dtype=str,
    encoding="utf-8-sig"
).fillna("")


# ============================================================
# CHECK REQUIRED COLUMNS
# ============================================================

required_goodreads_columns = [
    "Book Id",
    "Title",
    "Author",
    "ISBN",
    "My Rating",
    "Publisher",
    "Binding",
    "Number of Pages",
    "Publication Year",
    "Date Read",
    "Bookshelves",
    "Exclusive Shelf",
    "Review"
]

missing = [
    col for col in required_goodreads_columns
    if col not in goodreads.columns
]

if missing:
    raise ValueError(
        "The following Goodreads columns are missing:\n"
        + "\n".join(missing)
    )


required_book_columns = [
    "ImportBookID",
    "BookTitle",
    "Author",
    "ImportAuthorID",
    "Pages",
    "Format",
    "Fiction/Nonfiction",
    "Front/Back",
    "Genre",
    "OriginalLanguage",
    "ISBN",
    "PublicationYear",
    "OpenLibraryStatus"
]

missing = [
    col for col in required_book_columns
    if col not in books.columns
]

if missing:
    raise ValueError(
        "The following Books columns are missing:\n"
        + "\n".join(missing)
    )


if "Review" not in books.columns:
    books["Review"] = ""


books["_NormalizedTitle"] = books["BookTitle"].apply(normalize_title)
goodreads["_NormalizedTitle"] = goodreads["Title"].apply(normalize_title)


# ============================================================
# CREATE GOODREADS LOOKUP BY TITLE
# ============================================================

# We only want Goodreads records that have a title.
goodreads = goodreads[
    goodreads["_NormalizedTitle"] != ""
].copy()


# ============================================================
# PROCESS GOODREADS BOOK DATA
# ============================================================

print(f"Books in import file: {len(books):,}")
print(f"Rows in Goodreads export: {len(goodreads):,}")

matched_books = 0
unmatched_goodreads = 0
fields_filled = 0
reviews_added = 0


# Fields that Goodreads is allowed to fill when the existing import workbook field is blank.

goodreads_to_books = {
    "Number of Pages": "Pages",
    "Binding": "Format",
    "Publication Year": "PublicationYear",
    "ISBN": "ISBN",
}


for gr_index, gr_row in goodreads.iterrows():

    title_key = gr_row["_NormalizedTitle"]

    matches = books.index[
        books["_NormalizedTitle"] == title_key
    ].tolist()

    if not matches:
        unmatched_goodreads += 1
        continue

    # If there are duplicate titles in the import workbook,
    # use the first matching book.
    book_index = matches[0]

    matched_books += 1

    # Fill blank book fields only

    for gr_field, book_field in goodreads_to_books.items():

        gr_value = str(gr_row.get(gr_field, "")).strip()
        existing_value = str(
            books.at[book_index, book_field]
        ).strip()

        if gr_value and not existing_value:
            books.at[book_index, book_field] = gr_value
            fields_filled += 1

    # --------------------------------------------------------
    # REVIEW
    # --------------------------------------------------------

    review = str(gr_row.get("Review", "")).strip()

    if review:
        existing_review = str(
            books.at[book_index, "Review"]
        ).strip()

        if not existing_review:
            books.at[book_index, "Review"] = review
            reviews_added += 1


# ============================================================
# CREATE READ INSTANCES
# ============================================================

read_instance_rows = []

read_instances_added = 0


for gr_index, gr_row in goodreads.iterrows():

    title_key = gr_row["_NormalizedTitle"]

    if not title_key:
        continue

    matches = books.index[
        books["_NormalizedTitle"] == title_key
    ].tolist()

    if not matches:
        continue

    book_index = matches[0]

    import_book_id = books.at[
        book_index,
        "ImportBookID"
    ]

    book_title = books.at[
        book_index,
        "BookTitle"
    ]

    date_read = str(
        gr_row.get("Date Read", "")
    ).strip()

    # No Date Read = no read instance
    if not date_read:
        continue

    # --------------------------------------------------------
    # Convert Goodreads date to YYYY-MM-DD where possible
    # --------------------------------------------------------

    parsed_date = pd.to_datetime(
        date_read,
        errors="coerce"
    )

    if pd.notna(parsed_date):
        date_read_output = parsed_date.strftime("%Y-%m-%d")
        year_output = str(parsed_date.year)
    else:
        date_read_output = date_read
        year_output = ""


    # --------------------------------------------------------
    # Map Goodreads fields into Read Instances
    # --------------------------------------------------------

    rating = str(
        gr_row.get("My Rating", "")
    ).strip()

    fmt = str(
        gr_row.get("Binding", "")
    ).strip()


    new_instance = {
        "Book": import_book_id,
        "BookTitle": book_title,
        "DateRead": date_read_output,
        "Rating": rating,
        "Format": fmt,
        "LanguageRead": "",
        "Year": year_output,
        "Read": "Yes"
    }

    read_instance_rows.append(new_instance)


# ============================================================
# APPEND GOODREADS READ INSTANCES
# ============================================================

if read_instance_rows:

    goodreads_read_instances = pd.DataFrame(
        read_instance_rows
    )

    # Make sure columns are in the same order as the
    # existing Read Instances sheet.
    goodreads_read_instances = goodreads_read_instances[
        read_instances.columns
    ]

    read_instances = pd.concat(
        [
            read_instances,
            goodreads_read_instances
        ],
        ignore_index=True
    )

    read_instances_added = len(
        goodreads_read_instances
    )


# ============================================================
# ADD NEW GOODREADS BOOKS, AUTHORS, AND READ INSTANCES
# ============================================================
#
# This section ONLY handles Goodreads books that did not match
# an existing book in the import workbook.
#
# Existing Books, Authors, and Read Instances are not changed.
# ============================================================

new_books_added = 0
new_authors_added = 0
new_read_instances_added = 0


# ------------------------------------------------------------
# Helper: find the next available numeric ID
# ------------------------------------------------------------

def next_numeric_id(series, prefix=""):
    """
    Find the next available numeric ID.

    Works with values such as:
        1
        2
        Author001
        Author002
    """

    numbers = []

    for value in series.astype(str):
        value = value.strip()

        if not value:
            continue

        # Extract trailing number
        match = re.search(r"(\d+)$", value)

        if match:
            numbers.append(int(match.group(1)))

    next_number = max(numbers, default=0) + 1

    if prefix:
        return f"{prefix}{next_number}"

    return str(next_number)


# ------------------------------------------------------------
# Create lookup of existing authors
# ------------------------------------------------------------

def normalize_author(author):
    if pd.isna(author):
        return ""

    author = str(author).strip().lower()

    # Normalize whitespace
    author = re.sub(r"\s+", " ", author)

    return author


authors["_NormalizedAuthor"] = authors["Author"].apply(
    normalize_author
)


# ------------------------------------------------------------
# Create lookup of books already present
# ------------------------------------------------------------

existing_book_titles = set(
    books["_NormalizedTitle"].dropna().astype(str)
)

# This will also include books that were matched earlier in
# your program.


# ------------------------------------------------------------
# Process ONLY Goodreads books that are not already present
# ------------------------------------------------------------

for gr_index, gr_row in goodreads.iterrows():

    title_key = gr_row["_NormalizedTitle"]

    if not title_key:
        continue

    # Skip anything that already exists.

    if title_key in existing_book_titles:
        continue


    # --------------------------------------------------------
    # BASIC GOODREADS BOOK INFORMATION
    # --------------------------------------------------------

    title = str(
        gr_row.get("Title", "")
    ).strip()

    author_name = str(
        gr_row.get("Author", "")
    ).strip()

    isbn = str(
        gr_row.get("ISBN", "")
    ).strip()

    pages = str(
        gr_row.get("Number of Pages", "")
    ).strip()

    binding = str(
        gr_row.get("Binding", "")
    ).strip()

    publication_year = str(
        gr_row.get("Publication Year", "")
    ).strip()

    review = str(
        gr_row.get("Review", "")
    ).strip()


    # AUTHOR

    author_key = normalize_author(author_name)

    if author_key:

        if author_key in set(authors["_NormalizedAuthor"]):

            author_matches = authors.index[
                authors["_NormalizedAuthor"] == author_key
            ].tolist()

            author_index = author_matches[0]

            import_author_id = authors.at[
                author_index,
                "ImportAuthorID"
            ]

        else:

            import_author_id = next_numeric_id(
                authors["ImportAuthorID"],
                prefix=""
            )

            new_author = {
                "ImportAuthorID": import_author_id,
                "Author": author_name
            }

            # Add any other existing Authors columns as blank
            # so the new row has exactly the same structure.
            for column in authors.columns:
                if column not in new_author and column != "_NormalizedAuthor":
                    new_author[column] = ""

            authors = pd.concat(
                [
                    authors,
                    pd.DataFrame([new_author])
                ],
                ignore_index=True
            )

            # Update lookup immediately so another Goodreads
            # book by this author doesn't create another author.
            authors.loc[
                authors.index[-1],
                "_NormalizedAuthor"
            ] = author_key

            new_authors_added += 1


    else:
        import_author_id = ""


    # NEW BOOK ID

    import_book_id = next_numeric_id(
        books["ImportBookID"],
        prefix=""
    )


    # CREATE NEW BOOK

    new_book = {}

    for column in books.columns:

        if column == "_NormalizedTitle":
            continue

        new_book[column] = ""


    # Populate fields that can be determined from Goodreads
    new_book["ImportBookID"] = import_book_id
    new_book["BookTitle"] = title
    new_book["Author"] = author_name
    new_book["ImportAuthorID"] = import_author_id

    if pages:
        new_book["Pages"] = pages

    if binding:
        new_book["Format"] = binding

    if isbn:
        new_book["ISBN"] = isbn

    if publication_year:
        new_book["PublicationYear"] = publication_year

    if review and "Review" in books.columns:
        new_book["Review"] = review


    # Add the new book

    books = pd.concat(
        [
            books,
            pd.DataFrame([new_book])
        ],
        ignore_index=True
    )


    # Add to lookup so duplicate Goodreads rows don't create
    # another Books row.
    existing_book_titles.add(title_key)

    new_books_added += 1


# Remove temporary author lookup column

authors = authors.drop(
    columns=["_NormalizedAuthor"],
    errors="ignore"
)


# ============================================================
# ADD READ INSTANCES FOR NEW GOODREADS BOOKS
# ============================================================

# Rebuild the book lookup because new books have now been added.
books["_NormalizedTitle"] = books["BookTitle"].apply(
    normalize_title
)


for gr_index, gr_row in goodreads.iterrows():

    title_key = gr_row["_NormalizedTitle"]

    if not title_key:
        continue


    # Find the corresponding book.
    matches = books.index[
        books["_NormalizedTitle"] == title_key
    ].tolist()

    if not matches:
        continue

    book_index = matches[0]

    import_book_id = books.at[
        book_index,
        "ImportBookID"
    ]

    book_title = books.at[
        book_index,
        "BookTitle"
    ]


    # Only create a read instance if Goodreads has a Date Read

    date_read = str(
        gr_row.get("Date Read", "")
    ).strip()

    if not date_read:
        continue


    # Convert date where possible
    parsed_date = pd.to_datetime(
        date_read,
        errors="coerce"
    )

    if pd.notna(parsed_date):
        date_read_output = parsed_date.strftime("%Y-%m-%d")
        year_output = str(parsed_date.year)
    else:
        date_read_output = date_read
        year_output = ""


    rating = str(
        gr_row.get("My Rating", "")
    ).strip()

    binding = str(
        gr_row.get("Binding", "")
    ).strip()


    # Create a row using the EXACT columns in Read Instances

    new_instance = {}

    for column in read_instances.columns:
        new_instance[column] = ""


    # These are populated if these columns exist.
    if "Book" in new_instance:
        new_instance["Book"] = import_book_id

    if "BookTitle" in new_instance:
        new_instance["BookTitle"] = book_title

    if "DateRead" in new_instance:
        new_instance["DateRead"] = date_read_output

    if "Rating" in new_instance:
        new_instance["Rating"] = rating

    if "Format" in new_instance:
        new_instance["Format"] = binding

    if "Year" in new_instance:
        new_instance["Year"] = year_output

    if "Read" in new_instance:
        new_instance["Read"] = "Yes"


    read_instances = pd.concat(
        [
            read_instances,
            pd.DataFrame([new_instance])
        ],
        ignore_index=True
    )

    new_read_instances_added += 1


# Remove temporary title lookup

books = books.drop(
    columns=["_NormalizedTitle"],
    errors="ignore"
)

goodreads = goodreads.drop(
    columns=["_NormalizedTitle"],
    errors="ignore"
)


# ============================================================
# WRITE OUTPUT WORKBOOK
# ============================================================

with pd.ExcelWriter(
    output_file,
    engine="openpyxl"
) as writer:

    authors.to_excel(
        writer,
        sheet_name="Authors",
        index=False
    )

    books.to_excel(
        writer,
        sheet_name="Books",
        index=False
    )

    read_instances.to_excel(
        writer,
        sheet_name="Read Instances",
        index=False
    )


# ============================================================
# SUMMARY
# ============================================================

print(f"Books in import workbook:   {len(books):,}")
print(f"Rows in Goodreads export:   {len(goodreads):,}")
print(f"Matching Goodreads rows:    {matched_books:,}")
print(f"Unmatched Goodreads rows:   {unmatched_goodreads:,}")
print(f"Blank fields filled:        {fields_filled:,}")
print(f"Reviews added:              {reviews_added:,}")
print(f"Read instances added:      {read_instances_added:,}")
print()
print("New Goodreads records added:")
print(f"  New books:          {new_books_added:,}")
print(f"  New authors:        {new_authors_added:,}")
print(f"  New read instances: {new_read_instances_added:,}")
print()
print(f"Output file:")
print(output_file)

In [ ]:
import pandas as pd
import requests
import time
import re
import unicodedata
from difflib import SequenceMatcher
from pathlib import Path

# Files
input_file = Path("wordpress_book_database_import_goodreads.xlsx")
output_file = Path("wordpress_book_database_import_goodreads_isbn.xlsx")

# Read workbook
books = pd.read_excel(input_file, sheet_name="Books", dtype=str).fillna("")
authors = pd.read_excel(input_file, sheet_name="Authors", dtype=str).fillna("")
read_instances = pd.read_excel(input_file, sheet_name="Read Instances", dtype=str).fillna("")

books["ISBN"] = books["ISBN"].astype(str).str.strip()
missing_isbn = books[books["ISBN"].eq("")].copy()

print(f"Total books: {len(books)}")
print(f"Books missing ISBN: {len(missing_isbn)}\n")

# Open Library session
session = requests.Session()
session.headers.update({"User-Agent": "BookDatabaseISBNLookup/1.0"})


def normalize_text(text):
    replacements = {
        "Ã©": "é", "Ã¨": "è", "Ãª": "ê", "Ã«": "ë",
        "Ã¡": "á", "Ã¢": "â", "Ã£": "ã", "Ã¤": "ä",
        "Ã¥": "å", "Ã³": "ó", "Ã´": "ô", "Ã¶": "ö",
        "Ãº": "ú", "Ã¼": "ü", "Ã±": "ñ",
        "â€™": "'", "â€˜": "'", "â€œ": '"', "â€": '"',
        "â€“": "-", "â€”": "-", "\xa0": " "
    }

    text = str(text)
    for bad, good in replacements.items():
        text = text.replace(bad, good)

    text = unicodedata.normalize("NFKD", text)
    text = "".join(c for c in text if not unicodedata.combining(c))
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    return re.sub(r"\s+", " ", text).strip()


def clean_title(title):
    title = str(title).strip()

    # Remove Goodreads series information
    title = re.sub(r"\s*\([^)]*#\d+[^)]*\)\s*$", "", title)

    # Remove Goodreads date information
    title = re.sub(r"\s*\(\d{4}-\d{2}-\d{2}\)\s*$", "", title)

    return title.strip()


def simple_title(title):
    title = clean_title(title)

    # Remove subtitle for fallback search
    title = re.split(r"\s*:\s*", title)[0]
    title = re.split(r"\s+[–—-]\s+", title)[0]

    return title.strip()


def search_open_library(title, author=""):
    params = {
        "title": title,
        "fields": "key,title,author_name,isbn,isbn13,isbn10",
        "limit": 20
    }

    if author:
        params["author"] = author

    try:
        response = session.get(
            "https://openlibrary.org/search.json",
            params=params,
            timeout=30
        )
        response.raise_for_status()
        return response.json()
    except Exception as e:
        # print(f"    ERROR: {e}")
        return None


def valid_isbn13(isbn):
    isbn = str(isbn).strip()
    return len(isbn) == 13 and isbn.isdigit()


def valid_isbn10(isbn):
    isbn = str(isbn).strip().upper()
    return len(isbn) == 10 and isbn[:9].isdigit() and (
        isbn[9].isdigit() or isbn[9] == "X"
    )


def extract_isbn(docs):
    # Always look through ALL results for ISBN-13 first
    for doc in docs:
        for isbn in doc.get("isbn13", []) + doc.get("isbn", []):
            if valid_isbn13(isbn):
                return str(isbn).strip()

    # Only use ISBN-10 if no ISBN-13 exists
    for doc in docs:
        for isbn in doc.get("isbn10", []) + doc.get("isbn", []):
            if valid_isbn10(isbn):
                return str(isbn).strip().upper()

    return ""


def title_similarity(a, b):
    a, b = normalize_text(a), normalize_text(b)

    if a == b:
        return 1.0
    if a in b or b in a:
        return 0.90

    return SequenceMatcher(None, a, b).ratio()


def author_similarity(target, candidates):
    target = normalize_text(target)
    best = 0

    for candidate in candidates:
        candidate = normalize_text(candidate)

        if target == candidate:
            return 1.0
        if target in candidate or candidate in target:
            best = max(best, 0.90)
        else:
            best = max(
                best,
                SequenceMatcher(None, target, candidate).ratio()
            )

    return best


def best_document(data, title, author):
    if not data:
        return None

    best_doc = None
    best_score = 0

    for doc in data.get("docs", []):
        t_score = title_similarity(title, doc.get("title", ""))
        a_score = author_similarity(
            author,
            doc.get("author_name", [])
        )

        score = t_score * 0.65 + a_score * 0.35

        if score > best_score:
            best_score = score
            best_doc = doc

    return best_doc if best_score >= 0.55 else None


def lookup_isbn(title, author):
    cleaned = clean_title(title)
    simple = simple_title(title)

    # 1. Exact title + author
    data = search_open_library(title, author)
    isbn = extract_isbn(data.get("docs", [])) if data else ""
    if isbn:
        return isbn

    # 2. Cleaned title + author
    if cleaned != title:
        data = search_open_library(cleaned, author)
        isbn = extract_isbn(data.get("docs", [])) if data else ""
        if isbn:
            return isbn

    # 3. Simplified title + author
    if simple != cleaned:
        data = search_open_library(simple, author)
        isbn = extract_isbn(data.get("docs", [])) if data else ""
        if isbn:
            return isbn

    # 4. Title-only, with author verification
    data = search_open_library(cleaned)
    doc = best_document(data, cleaned, author)

    if doc:
        isbn = extract_isbn([doc])
        if isbn:
            return isbn

    # 5. Simplified title-only, with author verification
    if simple != cleaned:
        data = search_open_library(simple)
        doc = best_document(data, simple, author)

        if doc:
            isbn = extract_isbn([doc])
            if isbn:
                return isbn

    return ""


# Process missing ISBNs
found = 0
not_found = 0

for count, (index, row) in enumerate(missing_isbn.iterrows(), start=1):
    title = str(row["BookTitle"]).strip()
    author = str(row["Author"]).strip()

    # print(f"[{count}/{len(missing_isbn)}] {title} — {author}")

    if not title:
        not_found += 1
        # print("    Blank title")
        continue

    isbn = lookup_isbn(title, author)

    if isbn and books.at[index, "ISBN"] == "":
        books.at[index, "ISBN"] = isbn
        found += 1
        # print(f"    FOUND ISBN-{13 if len(isbn) == 13 else 10}: {isbn}")
    else:
        not_found += 1
        # print("    Not found")

    time.sleep(0.5)


# Save workbook
with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    authors.to_excel(writer, sheet_name="Authors", index=False)
    books.to_excel(writer, sheet_name="Books", index=False)
    read_instances.to_excel(writer, sheet_name="Read Instances", index=False)

# Summary
still_missing = books[books["ISBN"].eq("")]
isbn13_count = sum(books["ISBN"].apply(valid_isbn13))
isbn10_count = sum(books["ISBN"].apply(valid_isbn10))

print("\n" + "=" * 50)
print("ISBN LOOKUP COMPLETE")
print("=" * 50)
print(f"Total books:     {len(books)}")
print(f"Initially blank: {len(missing_isbn)}")
print(f"ISBNs found:     {found}")
print(f"ISBN-13 total:   {isbn13_count}")
print(f"ISBN-10 total:   {isbn10_count}")
print(f"Still missing:   {len(still_missing)}")
print(f"\nOutput file: {output_file}")

In [ ]:
import pandas as pd
import unicodedata
from pathlib import Path

input_file = Path("wordpress_book_database_import_goodreads_isbn.xlsx")
output_file = Path("wordpress_book_database_import_goodreads_isbn_cleaned.xlsx")

books = pd.read_excel(input_file, sheet_name="Books", dtype=str).fillna("")
authors = pd.read_excel(input_file, sheet_name="Authors", dtype=str).fillna("")
read_instances = pd.read_excel(input_file, sheet_name="Read Instances", dtype=str).fillna("")


def clean_unicode(text):
    replacements = {
        "Ã©": "é", "Ã¨": "è", "Ãª": "ê", "Ã«": "ë",
        "Ã¡": "á", "Ã¢": "â", "Ã£": "ã", "Ã¤": "ä",
        "Ã¥": "å", "Ã³": "ó", "Ã´": "ô", "Ã¶": "ö",
        "Ãº": "ú", "Ã¼": "ü", "Ã±": "ñ",
        "Ã§": "ç", "Ã‰": "É", "Ãˆ": "È",
        "â€™": "'", "â€˜": "'", "â€œ": '"', "â€": '"',
        "â€“": "–", "â€”": "—", "â€¦": "…",
        "\xa0": " "
    }

    text = str(text)
    for bad, good in replacements.items():
        text = text.replace(bad, good)

    return unicodedata.normalize("NFC", text).strip()


def clean_format(value):
    value = str(value).strip()

    format_map = {
        "Audio": "Audiobook",
        "Audio CD": "Audiobook",
        "Kindle Edition": "eBook",
        "Paper": "Print",
        "Paperback": "Print",
        "Hardcover": "Print",
        "Mass Market Paperback": "Print"
    }

    return format_map.get(value, value)


books["BookTitle"] = books["BookTitle"].apply(clean_unicode)
books["Author"] = books["Author"].apply(clean_unicode)
books["Format"] = books["Format"].apply(clean_format)

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    authors.to_excel(writer, sheet_name="Authors", index=False)
    books.to_excel(writer, sheet_name="Books", index=False)
    read_instances.to_excel(writer, sheet_name="Read Instances", index=False)

print("Cleaning complete.")
print(f"Output file: {output_file}")

In [ ]:
import pandas as pd
import requests
from pathlib import Path
from openpyxl import load_workbook


# ============================================================
# FILES
# ============================================================

input_file = Path("wordpress_book_database_import_goodreads_isbn_cleaned.xlsx")
output_file = Path("wordpress_book_database_import_goodreads_isbn_language_country.xlsx")


# ============================================================
# LOAD WORKBOOK
# ============================================================

books = pd.read_excel(
    input_file,
    sheet_name="Books",
    dtype=str
).fillna("")

authors = pd.read_excel(
    input_file,
    sheet_name="Authors",
    dtype=str
).fillna("")

read_instances = pd.read_excel(
    input_file,
    sheet_name="Read Instances",
    dtype=str
).fillna("")


# ============================================================
# SESSION
# ============================================================

session = requests.Session()

session.headers.update({
    "User-Agent": "BookDatabaseMetadataLookup/1.0"
})


# ============================================================
# LANGUAGE MAP
# ============================================================

language_map = {
    "eng": "English",
    "fra": "French",
    "fre": "French",
    "spa": "Spanish",
    "deu": "German",
    "ger": "German",
    "ita": "Italian",
    "por": "Portuguese",
    "jpn": "Japanese",
    "chi": "Chinese",
    "zho": "Chinese",
    "kor": "Korean",
    "rus": "Russian",
    "ara": "Arabic",
    "hin": "Hindi",
    "swe": "Swedish",
    "nor": "Norwegian",
    "dan": "Danish",
    "fin": "Finnish",
    "nld": "Dutch",
    "dut": "Dutch",
    "pol": "Polish",
    "tur": "Turkish",
    "heb": "Hebrew",
    "ell": "Greek",
    "gre": "Greek",
    "ces": "Czech",
    "cze": "Czech",
    "hun": "Hungarian",
    "ron": "Romanian",
    "rum": "Romanian",
    "ukr": "Ukrainian",
    "isl": "Icelandic",
    "ice": "Icelandic",
    "cat": "Catalan",
    "lat": "Latin",
    "san": "Sanskrit"
}


# ============================================================
# GET LANGUAGE FROM ISBN
# ============================================================

def get_book_language(isbn):

    isbn = str(isbn).strip().replace("-", "")

    if not isbn:
        return ""

    try:
        response = session.get(
            f"https://openlibrary.org/isbn/{isbn}.json",
            timeout=8
        )

        if response.status_code != 200:
            return ""

        data = response.json()

        for language in data.get("languages", []):

            code = language.get("key", "").split("/")[-1]

            if code in language_map:
                return language_map[code]

    except requests.RequestException:
        return ""

    except Exception:
        return ""

    return ""


# ============================================================
# GET AUTHOR COUNTRY
# ============================================================

def get_author_country(name):

    name = str(name).strip()

    if not name:
        return ""

    try:

        # Search for author
        response = session.get(
            "https://openlibrary.org/search/authors.json",
            params={
                "q": name,
                "limit": 5
            },
            timeout=8
        )

        if response.status_code != 200:
            return ""

        docs = response.json().get("docs", [])

        if not docs:
            return ""

        # Prefer exact name match
        name_lower = name.lower()

        author = None

        for doc in docs:

            if doc.get("name", "").strip().lower() == name_lower:
                author = doc
                break

        if author is None:
            author = docs[0]

        key = author.get("key", "")

        if not key:
            return ""

        # Get author details
        response = session.get(
            f"https://openlibrary.org{key}.json",
            timeout=8
        )

        if response.status_code != 200:
            return ""

        location = response.json().get("location", "")

        if isinstance(location, dict):
            location = location.get("value", "")

        return str(location).strip()

    except requests.RequestException:
        return ""

    except Exception:
        return ""


# ============================================================
# ORIGINAL LANGUAGE
# ============================================================

if "OriginalLanguage" not in books.columns:
    books["OriginalLanguage"] = ""

language_cache = {}

language_rows = books.index[
    books["OriginalLanguage"].str.strip().eq("")
    & books["ISBN"].str.strip().ne("")
]

print(f"Books needing language lookup: {len(language_rows)}")

language_found = 0

for count, index in enumerate(language_rows, start=1):

    isbn = books.at[index, "ISBN"].strip().replace("-", "")

    # Only query each ISBN once
    if isbn not in language_cache:
        language_cache[isbn] = get_book_language(isbn)

    language = language_cache[isbn]

    if language:
        books.at[index, "OriginalLanguage"] = language
        language_found += 1

    # Progress
    if count % 100 == 0:
        print(
            f"Language progress: {count}/{len(language_rows)}"
        )


# ============================================================
# COUNTRY
# ============================================================

if "Country" not in books.columns:
    books["Country"] = ""

country_cache = {}

country_rows = books.index[
    books["Country"].str.strip().eq("")
    & books["Author"].str.strip().ne("")
]

# Only unique authors need to be looked up
unique_authors = books.loc[
    country_rows, "Author"
].str.strip().drop_duplicates()

print(f"Unique authors needing country lookup: {len(unique_authors)}")

country_found = 0

for count, author in enumerate(unique_authors, start=1):

    country_cache[author] = get_author_country(author)

    if count % 50 == 0:
        print(
            f"Author progress: {count}/{len(unique_authors)}"
        )


# Apply cached country values to books
for index in country_rows:

    author = books.at[index, "Author"].strip()

    country = country_cache.get(author, "")

    if country:
        books.at[index, "Country"] = country
        country_found += 1


# ============================================================
# SAVE
# ============================================================

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:

    authors.to_excel(
        writer,
        sheet_name="Authors",
        index=False
    )

    books.to_excel(
        writer,
        sheet_name="Books",
        index=False
    )

    read_instances.to_excel(
        writer,
        sheet_name="Read Instances",
        index=False
    )


# ============================================================
# SUMMARY
# ============================================================

print()
print("========================================")
print("Lookup complete.")
print("========================================")
print(f"OriginalLanguage values added: {language_found}")
print(f"Country values added: {country_found}")
print(f"Unique ISBNs queried: {len(language_cache)}")
print(f"Unique authors queried: {len(country_cache)}")
print(f"Output file: {output_file}")

In [ ]:
import pandas as pd
import requests
from pathlib import Path
from openpyxl import load_workbook


# ============================================================
# FILES
# ============================================================

input_file = Path("wordpress_book_database_import_goodreads_isbn_cleaned.xlsx")
output_file = Path("wordpress_book_database_import_goodreads_isbn_language_country.xlsx")


# ============================================================
# LOAD WORKBOOK
# ============================================================

books = pd.read_excel(
    input_file,
    sheet_name="Books",
    dtype=str
).fillna("")

authors = pd.read_excel(
    input_file,
    sheet_name="Authors",
    dtype=str
).fillna("")

read_instances = pd.read_excel(
    input_file,
    sheet_name="Read Instances",
    dtype=str
).fillna("")


# ============================================================
# SESSION
# ============================================================

session = requests.Session()

session.headers.update({
    "User-Agent": "BookDatabaseMetadataLookup/1.0"
})


# ============================================================
# LANGUAGE MAP
# ============================================================

language_map = {
    "eng": "English",
    "fra": "French",
    "fre": "French",
    "spa": "Spanish",
    "deu": "German",
    "ger": "German",
    "ita": "Italian",
    "por": "Portuguese",
    "jpn": "Japanese",
    "chi": "Chinese",
    "zho": "Chinese",
    "kor": "Korean",
    "rus": "Russian",
    "ara": "Arabic",
    "hin": "Hindi",
    "swe": "Swedish",
    "nor": "Norwegian",
    "dan": "Danish",
    "fin": "Finnish",
    "nld": "Dutch",
    "dut": "Dutch",
    "pol": "Polish",
    "tur": "Turkish",
    "heb": "Hebrew",
    "ell": "Greek",
    "gre": "Greek",
    "ces": "Czech",
    "cze": "Czech",
    "hun": "Hungarian",
    "ron": "Romanian",
    "rum": "Romanian",
    "ukr": "Ukrainian",
    "isl": "Icelandic",
    "ice": "Icelandic",
    "cat": "Catalan",
    "lat": "Latin",
    "san": "Sanskrit"
}


# ============================================================
# GET LANGUAGE FROM ISBN
# ============================================================

def get_book_language(isbn):

    isbn = str(isbn).strip().replace("-", "")

    if not isbn:
        return ""

    try:
        response = session.get(
            f"https://openlibrary.org/isbn/{isbn}.json",
            timeout=8
        )

        if response.status_code != 200:
            return ""

        data = response.json()

        for language in data.get("languages", []):

            code = language.get("key", "").split("/")[-1]

            if code in language_map:
                return language_map[code]

    except requests.RequestException:
        return ""

    except Exception:
        return ""

    return ""


# ============================================================
# GET AUTHOR COUNTRY
# ============================================================

def get_author_country(name):

    name = str(name).strip()

    if not name:
        return ""

    try:

        # Search for author
        response = session.get(
            "https://openlibrary.org/search/authors.json",
            params={
                "q": name,
                "limit": 5
            },
            timeout=8
        )

        if response.status_code != 200:
            return ""

        docs = response.json().get("docs", [])

        if not docs:
            return ""

        # Prefer exact name match
        name_lower = name.lower()
        author = None

        for doc in docs:
            if doc.get("name", "").strip().lower() == name_lower:
                author = doc
                break

        if author is None:
            author = docs[0]

        key = author.get("key", "")

        if not key:
            return ""

        # Get author details
        response = session.get(
            f"https://openlibrary.org{key}.json",
            timeout=8
        )

        if response.status_code != 200:
            return ""

        data = response.json()

        # Open Library may store location as a string or dictionary
        location = data.get("location", "")

        if isinstance(location, dict):
            location = location.get("value", "")

        location = str(location).strip()

        if not location:
            return ""

        # ----------------------------------------------------
        # Convert known country names/codes to country only
        # ----------------------------------------------------

        country_map = {
            "canada": "Canada",
            "united states": "United States",
            "usa": "United States",
            "united states of america": "United States",
            "england": "United Kingdom",
            "scotland": "United Kingdom",
            "wales": "United Kingdom",
            "northern ireland": "United Kingdom",
            "united kingdom": "United Kingdom",
            "uk": "United Kingdom",
            "ireland": "Ireland",
            "france": "France",
            "germany": "Germany",
            "italy": "Italy",
            "spain": "Spain",
            "portugal": "Portugal",
            "japan": "Japan",
            "china": "China",
            "south korea": "South Korea",
            "korea": "South Korea",
            "india": "India",
            "russia": "Russia",
            "ukraine": "Ukraine",
            "australia": "Australia",
            "new zealand": "New Zealand",
            "sweden": "Sweden",
            "norway": "Norway",
            "denmark": "Denmark",
            "finland": "Finland",
            "netherlands": "Netherlands",
            "belgium": "Belgium",
            "switzerland": "Switzerland",
            "austria": "Austria",
            "poland": "Poland",
            "greece": "Greece",
            "israel": "Israel",
            "mexico": "Mexico",
            "brazil": "Brazil",
            "argentina": "Argentina",
            "chile": "Chile",
            "colombia": "Colombia",
            "south africa": "South Africa",
            "egypt": "Egypt"
        }

        location_lower = location.lower()

        # Look for a country name anywhere in the location
        for key_name, country in country_map.items():

            if key_name in location_lower:
                return country

        return ""

    except requests.RequestException:
        return ""

    except Exception:
        return ""

# ============================================================
# ORIGINAL LANGUAGE
# ============================================================

if "OriginalLanguage" not in books.columns:
    books["OriginalLanguage"] = ""

language_cache = {}

language_rows = books.index[
    books["OriginalLanguage"].str.strip().eq("")
    & books["ISBN"].str.strip().ne("")
]

print(f"Books needing language lookup: {len(language_rows)}")

language_found = 0

for count, index in enumerate(language_rows, start=1):

    isbn = books.at[index, "ISBN"].strip().replace("-", "")

    # Only query each ISBN once
    if isbn not in language_cache:
        language_cache[isbn] = get_book_language(isbn)

    language = language_cache[isbn]

    if language:
        books.at[index, "OriginalLanguage"] = language
        language_found += 1

    # Progress
    if count % 100 == 0:
        print(
            f"Language progress: {count}/{len(language_rows)}"
        )


# ============================================================
# COUNTRY
# ============================================================

if "Country" not in books.columns:
    books["Country"] = ""

country_cache = {}

country_rows = books.index[
    books["Country"].str.strip().eq("")
    & books["Author"].str.strip().ne("")
]

# Only unique authors need to be looked up
unique_authors = books.loc[
    country_rows, "Author"
].str.strip().drop_duplicates()

print(f"Unique authors needing country lookup: {len(unique_authors)}")

country_found = 0

for count, author in enumerate(unique_authors, start=1):

    country_cache[author] = get_author_country(author)

    if count % 50 == 0:
        print(
            f"Author progress: {count}/{len(unique_authors)}"
        )


# Apply cached country values to books
for index in country_rows:

    author = books.at[index, "Author"].strip()

    country = country_cache.get(author, "")

    if country:
        books.at[index, "Country"] = country
        country_found += 1


# ============================================================
# SAVE
# ============================================================

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:

    authors.to_excel(
        writer,
        sheet_name="Authors",
        index=False
    )

    books.to_excel(
        writer,
        sheet_name="Books",
        index=False
    )

    read_instances.to_excel(
        writer,
        sheet_name="Read Instances",
        index=False
    )


# ============================================================
# SUMMARY
# ============================================================

print()
print("========================================")
print("Lookup complete.")
print("========================================")
print(f"OriginalLanguage values added: {language_found}")
print(f"Country values added: {country_found}")
print(f"Unique ISBNs queried: {len(language_cache)}")
print(f"Unique authors queried: {len(country_cache)}")
print(f"Output file: {output_file}")

In [8]:
import sys
!{sys.executable} fix_unicode.py


10 value(s) fixed.
  Authors!B208                 'Tara Susman-PeÃ±a' -> 'Tara Susman-Peña'
  Authors!B221                 'Caroline Criado PÃ©rez' -> 'Caroline Criado Pérez'
  Authors!B1029                'Clarissa Pinkola EstÃ©s' -> 'Clarissa Pinkola Estés'
  Authors!B1063                'Kim ThÃºy' -> 'Kim Thúy'
  Authors!B1068                'Peter SÃ\xads' -> 'Peter Sís'
  Books!C3913                  'Peter SÃ\xads' -> 'Peter Sís'
  Read Instances!B1523         'La dame aux camÃ©lias' -> 'La dame aux camélias'
  Read Instances!B1533         'Harry Potter et le Prince de Sang-MÃªlÃ© (Harry Potter, #6)' -> 'Harry Potter et le Prince de Sang-Mêlé (Harry Potter, #6)'
  Read Instances!B1559         'The Children of HÃºrin' -> 'The Children of Húrin'
  Read Instances!B1562         "Moi, Malala, je lutte pour l'Ã©ducation et je rÃ©siste aux t" -> "Moi, Malala, je lutte pour l'éducation et je résiste aux tal"

Saved wordpress_book_database_import_review_fixed.xlsx
